# ISIC 2019 Silver validation

This notebook validates and normalizes ISIC 2019 Bronze data into Silver: label normalization, image decode/dimension validation, leakage-control groups, and rejected records.

Most of the actual logic lives in `data_platform.spark_io` as dataset-agnostic shared functions. This notebook mainly declares ISIC 2019's dataset-specific configuration (`data_platform.datasets.isic_2019.normalize_labels`, its input/output column names) and calls the shared functions in sequence, so a future dataset's Silver notebook can reuse them with its own configuration instead of re-deriving the same Spark plumbing.

Run `05_setup_tables_and_folders.ipynb` and `10_bronze_ingest.ipynb` before this notebook.


In [ ]:
%run ../_setup_env


In [ ]:
from data_platform.datasets.isic_2019 import normalize_labels
from data_platform.layout import build_layout, load_dataset_config, load_yaml_config
from data_platform.spark_io import (
    apply_label_normalization,
    assign_leakage_groups_and_write_inventory,
    build_accepted_rows,
    print_silver_outputs,
    reconcile_bronze_records,
    validate_images,
)

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
bronze_tables = layout["bronze_tables"]
silver_tables = layout["silver_tables"]

# The only dataset-specific configuration this notebook needs: the dataset's identity
# on the shared Silver tables, which raw Bronze columns feed label normalization, and
# the label column names normalize_labels produces. Everything else below is shared
# Spark plumbing from data_platform.spark_io.
DATASET_KEY = DATASET_CONFIG["dataset_key"]
DIAGNOSIS_INPUT_COLUMNS = ["diagnosis_1", "diagnosis_2", "diagnosis_3"]
LABEL_COLUMNS = ["malignancy", "specific_diagnosis"]

print(f"Bronze source metadata table: {bronze_tables['source_metadata']}")
print(f"Bronze image blob table: {bronze_tables['image_blobs']}")
print(f"Silver image inventory table: {silver_tables['image_inventory']}")
print(f"Silver leakage groups table: {silver_tables['leakage_groups']}")
print(f"Silver rejected records table: {silver_tables['rejected_records']}")


## Reconcile Bronze Records

`reconcile_bronze_records` rejects any Bronze image blob with no matching source metadata row before any label or image validation runs. Fully dataset-agnostic, shared via `data_platform.spark_io`.


In [ ]:
source_metadata_df = reconcile_bronze_records(spark, bronze_tables, silver_tables, DATASET_KEY)


## Normalize and Validate Labels

`apply_label_normalization` wraps ISIC 2019's `normalize_labels` as a Spark UDF, applies it to `DIAGNOSIS_INPUT_COLUMNS`, and rejects rows where none of `LABEL_COLUMNS` resolved. The orchestration is shared; `normalize_labels` and the column names are the only ISIC-2019-specific inputs.


In [ ]:
label_valid_df = apply_label_normalization(
    spark,
    source_metadata_df,
    normalize_labels,
    DIAGNOSIS_INPUT_COLUMNS,
    LABEL_COLUMNS,
    silver_tables,
    DATASET_KEY,
)


## Validate Images

`validate_images` left-joins label-valid rows to the Bronze image blob table (rejecting any with no matching blob), decodes and validates the rest via `data_platform.validate.decode_batch` batched through `mapInPandas`, and rejects decode/dimension failures. Fully dataset-agnostic, shared via `data_platform.spark_io`.


In [ ]:
image_valid_df = validate_images(spark, label_valid_df, bronze_tables, silver_tables, DATASET_KEY)


## Assign Leakage-Control Groups and Write Image Inventory

`build_accepted_rows` joins label-valid metadata to validated images into one accepted-row DataFrame, tagged with `DATASET_KEY`. `assign_leakage_groups_and_write_inventory` groups them (exact duplicates by `source_checksum` first, then `lesion_id`, then `patient_id`, otherwise a singleton — all scoped per dataset, and `group_id` embeds `dataset_key` so it stays globally unique even if two datasets happen to reuse the same lesion/patient/image IDs) and writes `silver.leakage_groups`/`silver.image_inventory` via idempotent `MERGE INTO`. Both are dataset-agnostic — only `LABEL_COLUMNS` and `DATASET_KEY` vary per dataset.


In [ ]:
accepted_df = build_accepted_rows(label_valid_df, image_valid_df, LABEL_COLUMNS, DATASET_KEY)
image_inventory_df, leakage_groups_df = assign_leakage_groups_and_write_inventory(spark, accepted_df, LABEL_COLUMNS, silver_tables)


## Review Outputs

`print_silver_outputs` shows accepted-vs-rejected summary stats for this run: rejection reasons, validation status, per-label-column distribution, and leakage-group sizes. Fully dataset-agnostic, shared via `data_platform.spark_io`.

In [ ]:
print_silver_outputs(spark, display, silver_tables, LABEL_COLUMNS, DATASET_KEY, image_inventory_df, leakage_groups_df)
